# IPOPT Test: Python + R

Minimal notebook to verify IPOPT is usable from both Python and R
in a Snowflake Workspace Notebook.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="ipopt_config.yaml")

## Install R ipoptr from GitHub

The `ipoptr` R package is not on CRAN or conda-forge. It compiles from
source against the IPOPT C library, which conda provides via `cyipopt`.
We set `PKG_CONFIG_PATH` so the `configure` script can find `libipopt`.

In [ ]:
import subprocess, os

conda_env = os.path.expanduser("~/.local/share/mamba/envs/workspace_env")

env = os.environ.copy()
env["PKG_CONFIG_PATH"] = f"{conda_env}/lib/pkgconfig:{env.get('PKG_CONFIG_PATH', '')}"
env["LD_LIBRARY_PATH"] = f"{conda_env}/lib:{env.get('LD_LIBRARY_PATH', '')}"
env["PATH"] = f"{conda_env}/bin:{env['PATH']}"

r_cmd = 'remotes::install_github("jyypma/ipoptr", force=TRUE, quiet=FALSE)'
result = subprocess.run(
    ["R", "--vanilla", "-e", r_cmd],
    env=env, capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
else:
    print("ipoptr installed successfully")

## Python: import cyipopt

Bridge the conda env's site-packages into the notebook kernel.

In [ ]:
import sys, os, glob

conda_env = os.path.expanduser("~/.local/share/mamba/envs/workspace_env")
os.environ["LD_LIBRARY_PATH"] = f"{conda_env}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

sp = glob.glob(f"{conda_env}/lib/python3.12/site-packages")
if sp:
    sys.path.insert(0, sp[0])
    print(f"Added {sp[0]} to sys.path")
else:
    print("WARNING: conda python3.12 site-packages not found")

import cyipopt
print(f"cyipopt {cyipopt.__version__} loaded successfully")

## Python: solve a simple problem with cyipopt

Minimise x1*x4*(x1+x2+x3) + x3 subject to constraints.

In [ ]:
import numpy as np
from cyipopt import minimize_ipopt

def objective(x):
    return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]

def constraint_eq(x):
    return x[0]*x[1]*x[2]*x[3] - 25.0

def constraint_ineq(x):
    return 40.0 - (x[0]**2 + x[1]**2 + x[2]**2 + x[3]**2)

x0 = np.array([1.0, 5.0, 5.0, 1.0])
bounds = [(1, 5)] * 4
constraints = [
    {"type": "eq", "fun": constraint_eq},
    {"type": "ineq", "fun": constraint_ineq},
]

result = minimize_ipopt(objective, x0, bounds=bounds, constraints=constraints)
print(f"Optimal: {result.x}")
print(f"Objective: {result.fun:.4f}")
print(f"Success: {result.success}")

## R: library(ipoptr)

Same problem solved from R using the ipoptr package.

In [ ]:
%%R
library(ipoptr)

eval_f <- function(x) x[1]*x[4]*(x[1]+x[2]+x[3]) + x[3]

eval_grad_f <- function(x) {
  c(x[4]*(2*x[1]+x[2]+x[3]),
    x[1]*x[4],
    x[1]*x[4] + 1,
    x[1]*(x[1]+x[2]+x[3]))
}

eval_g <- function(x) {
  c(x[1]*x[2]*x[3]*x[4],
    x[1]^2 + x[2]^2 + x[3]^2 + x[4]^2)
}

eval_jac_g <- function(x) {
  c(x[2]*x[3]*x[4], x[1]*x[3]*x[4], x[1]*x[2]*x[4], x[1]*x[2]*x[3],
    2*x[1], 2*x[2], 2*x[3], 2*x[4])
}

opts <- list("print_level" = 0, "tol" = 1e-7)

res <- ipoptr(
  x0 = c(1, 5, 5, 1),
  eval_f = eval_f,
  eval_grad_f = eval_grad_f,
  lb = rep(1, 4),
  ub = rep(5, 4),
  eval_g = eval_g,
  eval_jac_g = eval_jac_g,
  eval_jac_g_structure = list(c(1,2,3,4), c(1,2,3,4)),
  constraint_lb = c(25, -Inf),
  constraint_ub = c(Inf, 40),
  opts = opts
)

cat(sprintf("Optimal: [%.4f, %.4f, %.4f, %.4f]\n",
            res$solution[1], res$solution[2], res$solution[3], res$solution[4]))
cat(sprintf("Objective: %.4f\n", res$objective))
cat(sprintf("Status: %d\n", res$status))